# 10 — Monitoring pack (the Board-style MI dashboard)

**Plain English:** The final step assembles `outputs/report/monitoring_pack.md`
from the real outputs of every prior notebook. It is built to read as a **Board
monitoring pack**: it **opens with a RAG dashboard** tied to the appetite limits
(notebook 07) and an **actions table** for anything amber/red, then lays out the
supporting evidence — each metric labelled **leading or lagging** — and closes
with the governance, stress and disclosure notes.

> Format only — illustrative, **not a regulatory submission**.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
pd.set_option("display.width", 120); pd.set_option("display.max_columns", 30)
import monitor as m
print("monitor library loaded — vintages:", m.VINTAGES)

monitor library loaded — vintages: ['2007', '2008', '2015']


In [2]:
# Assemble the Board-style monitoring pack from real outputs -------------
T = m.OUT_TABLES
rep = m.REPO_ROOT / "outputs" / "report"
rep.mkdir(parents=True, exist_ok=True)

appetite = pd.read_csv(T / "07_appetite_status.csv")
trend    = pd.read_csv(T / "07_leading_trends.csv")
stress   = pd.read_csv(T / "07_stress_vs_limits.csv")
probs_b  = pd.read_csv(T / "02_bucket_transition_matrix.csv", index_col=0)
rr       = pd.read_csv(T / "02_roll_rates.csv")
moves    = pd.read_csv(T / "03_stage_movements.csv")
wsumm    = pd.read_csv(T / "04_watchlist_summary.csv")
vint     = pd.read_csv(T / "05_vintage_cumulative_default.csv")
state    = pd.read_csv(T / "06_concentration_state.csv", index_col=0)
hhi_tbl  = pd.read_csv(T / "06_concentration_hhi.csv")
lvr      = pd.read_csv(T / "06_concentration_lvr.csv", index_col=0)
mod_view = pd.read_csv(T / "08_modified_exposure.csv")
scal     = pd.read_csv(T / "08_collections_scalability.csv")
psi_tbl  = pd.read_csv(T / "09_psi.csv")
grade    = pd.read_csv(T / "09_realised_default_by_grade.csv", index_col=0)

worst = appetite.RAG.map({"GREEN": 0, "AMBER": 1, "RED": 2, "n/a": 0}).max()
overall = {0: "GREEN", 1: "AMBER", 2: "RED"}[int(worst)]

L = []
def add(*xs): L.extend(xs)

add("# Portfolio Monitoring Pack — loan-level (Freddie Mac SFLLD)\n")
add("_Real loan-level mortgage data. The monitoring mechanics apply equally to "
    "commercial loan portfolios with a monthly status feed._\n")
add("_Format only — illustrative, **not a regulatory submission**._\n")

# --- 1. RAG dashboard (MON-2) ------------------------------------------------
add("## 1. Risk appetite dashboard — RAG vs limits\n")
add(f"**Overall portfolio status: {overall}.** Each metric is scored against the "
    "amber (early-warning) and red (limit) thresholds in `config/risk_appetite.yaml` "
    "(APS 220 paras 20/35; APG 220 para 65). `type` flags leading vs lagging.\n")
dash = appetite[["metric", "type", "last_period", "this_period", "amber", "red (limit)", "RAG"]]
add(dash.to_markdown(index=False) + "\n")

# Actions table for anything amber/red
flagged = appetite[appetite.RAG.isin(["AMBER", "RED"])]
add("### Actions (amber/red)\n")
if len(flagged):
    act = flagged[["metric", "RAG", "breach_action", "owner", "review_cycle"]].rename(
        columns={"breach_action": "action", "review_cycle": "due"})
    add(act.to_markdown(index=False) + "\n")
else:
    add("_All metrics within appetite this period — no escalations required._\n")
add("**Forward-looking view:** leading indicators (Stage 2 share, roll rates, SICR) "
    "are read first because they move before losses; the vintage curves (section 7) show "
    "how fast a downturn cohort can deteriorate, and the stress test (section 10) shows "
    "the same metrics against their limits under a downturn multiple.\n")

# --- 2. Leading-indicator trends (MON-3) ------------------------------------
add("## 2. Leading-indicator trends (forward-looking)\n")
add("APG 220 para 66 — do not rely solely on lagging arrears. Trailing-12m roll rates "
    "and the SICR (Current->Stage 2) migration rate, tracked over time:\n")
add(trend.to_markdown(index=False) + "\n")

# --- 3. Transition matrix + roll rates (lagging/leading labelled) -----------
add("## 3. Monthly delinquency-bucket transition matrix  _(lagging)_\n")
add(probs_b.round(4).to_markdown() + "\n")
add("![heatmap](../charts/02_bucket_transition_heatmap.png)\n")
add("## 4. Headline roll rates  _(leading — deterioration moves before default)_\n")
add(rr.to_markdown(index=False) + "\n")

# --- 5. Stage movements / 6. watchlist / 7. vintage -------------------------
add("## 5. IFRS 9 stage movements (loan-months)  _(mixed)_\n")
add(moves.to_markdown(index=False) + "\n")
add("## 6. Early-warning watchlist (by vintage / stage)  _(leading)_\n")
add(wsumm.to_markdown(index=False) + "\n")
add("## 7. Vintage tracking — cumulative default by months on book  _(lagging)_\n")
add(vint.to_markdown(index=False) + "\n")
add("![vintage curves](../charts/05_vintage_default_curves.png)\n")

# --- 8. Concentration: state + HHI + LVR (MON-4) ----------------------------
add("## 8. Concentration — geography, HHI & high-LVR (APS 220 para 35)\n")
add("_Format only — illustrative, not a regulatory submission._\n")
add("**By state (top 10):**\n")
add(state.head(10).round(2).to_markdown() + "\n")
add("**Geographic HHI:**\n")
add(hhi_tbl.to_markdown(index=False) + "\n")
add("**High-LVR concentration (by original LVR band):**\n")
add(lvr.round(2).to_markdown() + "\n")

# --- 9. Problem exposures (MON-5) -------------------------------------------
add("## 9. Problem exposures — modifications & collections scalability (APS 220 para 79)\n")
add("Modified / restructured loans and whether they cured or re-defaulted:\n")
add(mod_view.to_markdown(index=False) + "\n")
add("Collections scalability — trough vs crisis-peak monthly arrears (30+DPD) rate; "
    "the surge multiple is the load the workout function must be able to absorb:\n")
add(scal.to_markdown(index=False) + "\n")

# --- 10. Model performance / PSI (MON-6) ------------------------------------
add("## 10. Model performance — population stability (PSI) & backtest feed\n")
add("Layer 4 (rating-system performance). PSI of origination features vs the calm-2015 "
    "reference, and realised default by grade — the backtest feed for the sister "
    "[mortgage-credit-risk-pd-lgd-ead](https://github.com/Jane511/mortgage-credit-risk-pd-lgd-ead) model:\n")
add(psi_tbl.to_markdown(index=False) + "\n")
add("Realised cumulative default (%) by credit-score grade x vintage:\n")
add(grade.to_markdown() + "\n")

# --- 11. Governance / stress / disclosure notes (MON-7/8/9) -----------------
add("## 11. Governance, stress & disclosure notes\n")
add("**Stress -> limits (MON-7; APS 220 para 73 / APG 220 para 76).** Applying a "
    "downturn multiple (grounded in this repo's own vintage curves — 2007 reaches ~4x "
    "2015 default) to the flow/quality metrics re-tests them against their limits:\n")
add(stress.to_markdown(index=False) + "\n")
add("**Governance & independent validation (MON-8; APS 220 paras 28/75-76; APG 113 para 140).** "
    "Reporting cadence: the watchlist and roll rates go monthly to the Credit Risk Committee; "
    "the appetite RAG dashboard and concentration go monthly to the Board Risk Committee; "
    "the PSI/model-performance layer goes at least annually to model governance. The monitoring "
    "framework itself would be **independently validated annually**. _Demo, not a production system._\n")
add("**APS 330 / Pillar 3 framing (MON-9).** The concentration and credit-quality outputs "
    "(sections 7-8) are the inputs that feed **Pillar 3 (APS 330)** credit-risk disclosure. "
    "Any APS 330-style table here is **format only — illustrative, not a regulatory submission**.\n")

(rep / "monitoring_pack.md").write_text("\n".join(L), encoding="utf-8")
print("wrote ->", rep / "monitoring_pack.md", f"| overall RAG: {overall}")

wrote -> D:\Jane\Job Search\Github\bank\github project\mortgage-portfolio-monitoring\outputs\report\monitoring_pack.md | overall RAG: GREEN
